# 08 · PLSR vs LightGBM + streaming serving

Same features (`sqrt(PSD)/speed`), two backends, one pluggable interface.
Then we save a model artifact and run the **streaming inference** path that the
serving app will use: one recording in -> prediction, or a rejection with the
QC reason.

Key idea: the saved model **bundles its preprocessing contract** (cfg, feature
columns, transform, training speed range), so serving can't drift from training.


In [4]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from intelliswitch import Config, paths

# --- paths -----------------------------------------------------------------
# The project data lake (medallion tiers) is resolved by intelliswitch.paths, so
# it works regardless of the notebook's working dir and is overridable via the
# INTELLISWITCH_DATA_DIR / INTELLISWITCH_MODELS_DIR env vars.
RAW_CACHE = paths.RAW_CACHE       # data/interim/raw_cache  (decoded .npz, stage 1)
STORE     = paths.FEATURE_STORE   # data/processed/feature_store
MODELS    = paths.MODELS          # data/models
paths.ensure()                    # create the lake dirs if missing

# Raw source recordings (~7,800 logs) are large and live on the external drive;
# point DATA_ROOT at them, or symlink/copy the source into paths.RAW.
DATA_ROOT = Path(r"/Volumes/Seagate Expansion Drive/Mohammad/M Drive DTU/MATLAB Scripts/IntelliswitchScripts")
W10   = DATA_ROOT / "W10v140"       # vibration .log
BLADE = DATA_ROOT / "Bladenr8252"   # geometry .csv

cfg = Config()
cfg.cp_decimate = 1

## Load features, select trains, attach labels


In [5]:
from intelliswitch.pipeline import load_feature_store, select_trains
from intelliswitch.geometry import align_campaigns, build_deg_model
from intelliswitch.geometry.deg_model import attach_labels

store = load_feature_store(STORE)
feat = select_trains(store, cfg)
deg = build_deg_model(align_campaigns(BLADE, cfg), cfg)
data = attach_labels(feat, deg, cfg).dropna(subset=['y'])
print('rows:', len(data))


rows: 32891


In [6]:
feat.shape, store.shape

((32891, 1001), (32982, 1001))

In [26]:
feat.index

RangeIndex(start=0, stop=32891, step=1)

In [22]:
store.days.value_counts().reset_index().sort_values('days')

,days,count
592,-901.0,2
521,-900.0,9
448,-899.0,17
376,-898.0,25
522,-897.0,9
...,...,...
561,40.0,6
359,41.0,28
364,42.0,27
365,43.0,27


## Train both backends on the same data and compare
Overall test-point metrics, plus leave-one-location-out (generalization).


In [8]:
from intelliswitch.model import train, evaluate, leave_one_location_out

rows = []
for kind in ['plsr', 'lgbm']:
    m = train(data, cfg, kind)
    ov = evaluate(m, data, cfg)['overall']
    lolo = leave_one_location_out(data, cfg, kind)
    lolo_corr = np.nanmean([e['corr'] for e in lolo.values() if 'corr' in e])
    rows.append(dict(model=kind, info=m.info, n_train=m.n_train,
                     train_R2=round(m.train_r2,3),
                     n_test=ov['n_test'],
                     test_corr=round(ov['corr'],3), test_RMSE=round(ov['rmse'],3),
                     LOLO_corr=round(float(lolo_corr),3)))
pd.DataFrame(rows)


,model,info,n_train,train_R2,n_test,test_corr,test_MAPE,LOLO_corr
0,plsr,"{'backend': 'plsr', 'n_components': 18}",14026,0.562,7,-0.406,3.2,0.136
1,lgbm,"{'backend': 'lgbm', 'n_estimators': 300}",14026,0.962,7,-0.273,2.2,-0.118


Pick the winner by **LOLO correlation** (generalization to unseen locations),
not train R^2 — LightGBM will happily overfit 991 features.


In [9]:
ov

{'n_test': 7,
 'mape': 2.161036148117389,
 'corr': -0.2728936966890246,
 'pred': array([-1.45823409, -1.53163485, -1.48647665, -1.47630561, -1.51320564,
        -1.47871769, -1.49847163]),
 'ref': array([-1.5628267 , -1.51407312, -1.49192358, -1.49186658, -1.49703524,
        -1.50238875, -1.54818306]),
 'days': array([-882.5, -684.5, -444.5, -422. , -325. , -240. , -165. ])}

## Save the chosen model artifact


In [10]:
from intelliswitch.model import QualityModel
best_kind = 'plsr'    # set from the comparison above
model = train(data, cfg, best_kind)
MODELS.mkdir(parents=True, exist_ok=True)
path = model.save(str(MODELS / 'quality_model.joblib'))
print('saved', path, '| kind', model.kind, '| speed range', model.train_speed_range)


saved /Users/Mohammad/Documents/ds-projects/intelliswitch-refactor/data/models/quality_model.joblib | kind plsr | speed range (139.05275577279622, 165.20375129326172)


## Streaming inference — the serving entry point
One recording in -> predictions at three levels:
**wheel-set -> location -> S&C** (see `cfg.agg_wheelsets`, `cfg.agg_locations`).
Location-level is the informative output; the S&C number is a convenience
roll-up. The model carries its own cfg, so we re-specify no preprocessing.


In [11]:
from intelliswitch.io import load_raw
from intelliswitch.serving import predict_recording
from pathlib import Path

model = QualityModel.load(str(MODELS / 'quality_model.joblib'))
npz = sorted(RAW_CACHE.glob('*.npz'))

shown_ok = shown_rej = False
for p in npz[7800:]:
    res = predict_recording(load_raw(p), model)
    if res.ok and len(res.locations) >= 2:
        print('OK      ', res.source, '| %d wheel-sets / %d locations'
              % (res.meta['n_wheelsets'], res.meta['n_locations']),
              '| OOD=%s' % res.out_of_domain)
        for loc in res.locations:
            ws = ', '.join('g%d=%.2f' % (w.group_idx, w.quality) for w in loc.wheelsets)
            print('   LOCATION %-3s q=%.3f  <- %s of [%s]'
                  % (loc.sensor, loc.quality, model.cfg.agg_wheelsets, ws))
        print('   S&C indicator = %s of locations = %.3f'
              % (model.cfg.agg_locations, res.sc_quality))
        shown_ok = True
    elif (not res.ok) and not shown_rej:
        print('REJECTED', res.source, '->', res.reason)
        print('          app: "prediction not possible on this input"')
        shown_rej = True
    


REJECTED 190210_062750.log -> n_groups 5 != required (4, 5)
          app: "prediction not possible on this input"


In [12]:
len(npz)

7823

## Batch a stream and summarize (serving monitoring view)
Coverage = fraction of recordings we could serve; the rest are rejected with
reasons you'd log and alert on.


In [13]:
from collections import Counter
results = [predict_recording(load_raw(p), model) for p in npz[3000:3020]]
ok = [r for r in results if r.ok]
print(f'served {len(ok)}/{len(results)} ({100*len(ok)/len(results):.0f}%)  |',
      f'OOD flagged: {sum(r.out_of_domain for r in ok)}')
print('reject reasons:', dict(Counter(r.reason for r in results if not r.ok)))
agg = [r.aggregate for r in ok]
print('predicted quality: min=%.2f med=%.2f max=%.2f' % (np.min(agg), np.median(agg), np.max(agg)))


served 0/20 (0%)  | OOD flagged: 0
reject reasons: {'n_groups 5 != required (4, 5)': 19, 'n_groups 4 != required (4, 5)': 1}


ValueError: zero-size array to reduction operation minimum which has no identity